# Model Comparison in NAMpy

This notebook compares different NAMpy models for regression tasks and provides guidance on when to use each model variant.

## Available Models

NAMpy provides several model architectures:

- **NAM** (Neural Additive Model): Standard feedforward networks for each feature
- **NBM** (Neural Basis Model): Uses basis functions for smoother shape functions
- **GPNAM**: Incorporates Gaussian Process priors
- **NAMformer**: Uses transformer architecture for feature networks
- **NATT**: Neural Attention model
- **NodeGAM**: Tree-based neural network approach

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import time
import warnings
warnings.filterwarnings('ignore')

# Import all NAMpy regressor models
from nampy.models import (
    NAMRegressor,
    NBMRegressor,
    GPNAMRegressor,
    NAMformerRegressor,
    NATTRegressor,
    NodeGAMRegressor,
)

np.random.seed(42)

## 1. Prepare Data

We'll use the California Housing dataset for a real-world comparison.

In [ ]:
# Load data
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

# Use a subset for faster training
X_subset, _, y_subset, _ = train_test_split(
    X, y, train_size=5000, random_state=42
)

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_subset, y_subset, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {list(X.columns)}")

## 2. Define Models

We'll create instances of each model with comparable configurations.

In [ ]:
# Common training parameters
train_params = {
    'max_epochs': 50,
    'lr': 1e-3,
    'patience': 5,
    'batch_size': 128,
}

# Define models with comparable configurations
models = {
    'NAM': NAMRegressor(
        numerical_preprocessing='ple',
        n_bins=30,
        dropout=0.1,
        layer_sizes=[64, 32],
    ),
    'NBM': NBMRegressor(
        numerical_preprocessing='ple',
        n_bins=30,
        dropout=0.1,
    ),
    'NAMformer': NAMformerRegressor(
        numerical_preprocessing='ple',
        n_bins=30,
        dropout=0.1,
    ),
    'NATT': NATTRegressor(
        numerical_preprocessing='ple',
        n_bins=30,
        dropout=0.1,
    ),
    'NodeGAM': NodeGAMRegressor(
        numerical_preprocessing='ple',
        n_bins=30,
    ),
}

print(f"Models to compare: {list(models.keys())}")

## 3. Train and Evaluate Models

In [ ]:
# Train each model and collect results
results = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    print(f"{'='*50}")
    
    start_time = time.time()
    
    try:
        model.fit(X_train, y_train, **train_params)
        training_time = time.time() - start_time
        
        # Predict
        y_pred = model.predict(X_test)
        
        # Calculate metrics
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        results[name] = {
            'MSE': mse,
            'RMSE': rmse,
            'MAE': mae,
            'R²': r2,
            'Training Time (s)': training_time,
            'model': model
        }
        
        print(f"  R² = {r2:.4f}")
        print(f"  RMSE = {rmse:.4f}")
        print(f"  Training time: {training_time:.1f}s")
        
    except Exception as e:
        print(f"  Error: {str(e)}")
        results[name] = {
            'MSE': np.nan,
            'RMSE': np.nan,
            'MAE': np.nan,
            'R²': np.nan,
            'Training Time (s)': np.nan,
            'model': None
        }

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame({k: {m: v for m, v in val.items() if m != 'model'} 
                           for k, val in results.items()}).T

# Sort by R² (descending)
results_df = results_df.sort_values('R²', ascending=False)

print("\n" + "="*60)
print("MODEL COMPARISON RESULTS")
print("="*60)
print(results_df.to_string())

## 4. Visualize Results

In [ ]:
# Filter out models that failed
valid_results = results_df.dropna()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# R² Score
ax = axes[0, 0]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(valid_results)))
bars = ax.bar(valid_results.index, valid_results['R²'], color=colors, edgecolor='black')
ax.set_ylabel('R² Score')
ax.set_title('R² Score (Higher is Better)')
ax.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, valid_results['R²']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# RMSE
ax = axes[0, 1]
bars = ax.bar(valid_results.index, valid_results['RMSE'], color=colors, edgecolor='black')
ax.set_ylabel('RMSE')
ax.set_title('RMSE (Lower is Better)')
ax.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, valid_results['RMSE']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# MAE
ax = axes[1, 0]
bars = ax.bar(valid_results.index, valid_results['MAE'], color=colors, edgecolor='black')
ax.set_ylabel('MAE')
ax.set_title('Mean Absolute Error (Lower is Better)')
ax.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, valid_results['MAE']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Training Time
ax = axes[1, 1]
bars = ax.bar(valid_results.index, valid_results['Training Time (s)'], color=colors, edgecolor='black')
ax.set_ylabel('Time (seconds)')
ax.set_title('Training Time (Lower is Better)')
ax.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, valid_results['Training Time (s)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            f'{val:.1f}s', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Compare Shape Functions

Let's compare how different models learn shape functions for the same feature.

In [ ]:
# Select feature to compare
feature_idx = 0  # MedInc (Median Income)
feature_name = housing.feature_names[feature_idx]

# Get shape functions for each model
shape_functions = {}
for name, res in results.items():
    if res['model'] is not None:
        try:
            shape_out = res['model'].get_shape_function_outputs(X_test)
            shape_functions[name] = shape_out[:, feature_idx]
        except Exception as e:
            print(f"Could not get shape function for {name}: {e}")

In [ ]:
# Plot shape functions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

sort_idx = np.argsort(X_test[feature_name].values)
x_sorted = X_test[feature_name].values[sort_idx]

for ax, (name, shape) in zip(axes, shape_functions.items()):
    y_sorted = shape[sort_idx]
    ax.scatter(x_sorted, y_sorted, alpha=0.3, s=10)
    ax.set_xlabel(feature_name)
    ax.set_ylabel('Shape Function Output')
    ax.set_title(f'{name}')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Hide unused subplots
for i in range(len(shape_functions), len(axes)):
    axes[i].axis('off')

plt.suptitle(f'Shape Functions for "{feature_name}" Across Models', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Overlay all shape functions
plt.figure(figsize=(12, 6))

colors = plt.cm.tab10(np.linspace(0, 1, len(shape_functions)))

for (name, shape), color in zip(shape_functions.items(), colors):
    y_sorted = shape[sort_idx]
    # Smooth using rolling mean
    window = 20
    y_smooth = pd.Series(y_sorted).rolling(window, center=True).mean().values
    plt.plot(x_sorted, y_smooth, label=name, color=color, lw=2, alpha=0.8)

plt.xlabel(feature_name)
plt.ylabel('Shape Function Output')
plt.title(f'Shape Function Comparison for "{feature_name}"')
plt.legend()
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 6. Model Selection Guide

Based on the comparison, here's guidance on when to use each model:

| Model | Best For | Pros | Cons |
|-------|----------|------|------|
| **NAM** | General purpose | Simple, fast, interpretable | May overfit with many features |
| **NBM** | Smooth functions | Smoother shape functions | More parameters |
| **GPNAM** | Uncertainty quantification | Provides uncertainty estimates | Slower training |
| **NAMformer** | Complex patterns | Captures complex relationships | Higher memory usage |
| **NATT** | Feature interactions | Attention mechanism | May be harder to interpret |
| **NodeGAM** | Tree-like patterns | Works well with discrete features | May need tuning |

In [ ]:
# Print final recommendations
print("\n" + "="*60)
print("RECOMMENDATIONS")
print("="*60)

best_r2 = valid_results['R²'].idxmax()
best_time = valid_results['Training Time (s)'].idxmin()

print(f"\n🏆 Best Performance (R²): {best_r2} ({valid_results.loc[best_r2, 'R²']:.4f})")
print(f"⚡ Fastest Training: {best_time} ({valid_results.loc[best_time, 'Training Time (s)']:.1f}s)")

# Calculate efficiency score (R² / training_time)
valid_results['Efficiency'] = valid_results['R²'] / valid_results['Training Time (s)']
best_efficiency = valid_results['Efficiency'].idxmax()
print(f"⚖️ Best Efficiency (R²/time): {best_efficiency}")

## Summary

In this notebook, we:

1. **Compared multiple NAMpy models** on the same dataset
2. **Evaluated performance** using R², RMSE, and MAE metrics
3. **Measured training efficiency** (time vs. performance)
4. **Visualized shape functions** to compare interpretability
5. **Provided model selection guidance** based on use cases

The best model depends on your specific requirements:
- **Accuracy**: Choose the model with highest R²
- **Speed**: Choose NAM for fastest training
- **Interpretability**: All models provide shape functions, but some are smoother
- **Uncertainty**: Use GPNAM or LSS variants for uncertainty quantification